# The Boundary Lever

## 02 — Data

Everything computed from the Cayley-Dickson table in
`ValaQuenta/modules/box_kite/maths.py`. Nothing is read in from the literature;
agreement with published counts is a **check, not an input**.

⚠ **Sections 2.1-2.3 are ORIGINATING OBSERVATIONS**, computed before the paper
existed. They are what suggested the claim. They are **not** tests of it.
The tests are in `03_results.ipynb`.


In [1]:
import sys, os, math, itertools
sys.path.insert(0, os.path.abspath('/home/rendier/Projects/ThePlace'))
import numpy as np
from ValaQuenta.modules.box_kite.maths import cd_multiplication_table
np.set_printoptions(linewidth=120)


In [2]:
def cd(level):
    """Cayley-Dickson table and dimension at a doubling level."""
    return cd_multiplication_table(level)

def basis_ops(tab, dim):
    """P[k] = the (dim x dim) matrix of LEFT multiplication by e_k.

    Precomputing these once turns every L_a for a basis-pair divisor into a
    single matrix add, instead of a dim^2 Python loop per candidate. Without
    this the dim=128 sweep is hours; with it, seconds."""
    P = np.zeros((dim, dim, dim))
    for i in range(dim):
        for j in range(dim):
            s, k = tab[(i, j)]        # box_kite returns (SIGN, INDEX)
            P[i, k, j] = s
    return P

def diagonal_L(P, i, j, sgn):
    """L_a for a = (e_i + sgn*e_j)/sqrt(2)."""
    return (P[i] + sgn * P[j]) / math.sqrt(2.0)

def zd_census(level, tol=1e-10, verbose=False):
    """Every (i<j, sign) whose diagonal is a zero divisor, with its nullity."""
    tab, dim = cd(level); H = dim // 2
    P = basis_ops(tab, dim)
    rows = []
    for i in range(dim):
        for j in range(i + 1, dim):
            for sgn in (1.0, -1.0):
                s = np.linalg.svd(diagonal_L(P, i, j, sgn), compute_uv=False)
                n = int((s < tol).sum())
                if n:
                    loc = 'lower' if j < H else ('upper' if i >= H else 'cross')
                    rows.append({'i': i, 'j': j, 'sign': int(sgn), 'nullity': n, 'loc': loc})
    return dim, H, rows

def orphans_from(dim, rows):
    used = {r['i'] for r in rows} | {r['j'] for r in rows}
    return [k for k in range(dim) if k not in used]

def mirror_test(level):
    """Is e_(i+H) e_(j+H) the lower product with the order REVERSED?"""
    tab, dim = cd(level); H = dim // 2
    idx = [k for k in range(1, H)]
    rev = fwd = tot = 0
    for i in idx:
        for j in idx:
            if i == j: continue
            up = tab[(i + H, j + H)]
            if up == tab[(j, i)]: rev += 1
            if up == tab[(i, j)]: fwd += 1
            tot += 1
    esc = sum(1 for i in range(H) for j in range(H) if tab[(i + H, j + H)][1] >= H)
    return {'dim': dim, 'H': H, 'reversed': rev, 'preserved': fwd, 'pairs': tot,
            'upper_escapes_lower': esc, 'upper_pairs': H * H}


## 2.1 The mirror — ORIGINATING OBSERVATION

Is `e_(i+H) e_(j+H)` the lower product with the order preserved, or reversed?


In [3]:
print(f"{'dim':>5} {'boundary':>10} {'REVERSED':>16} {'preserved':>12} {'upper escapes':>16}")
for lv in (3, 4, 5, 6):
    m = mirror_test(lv)
    print(f"{m['dim']:>5} {'e_'+str(m['H']):>10} "
          f"{str(m['reversed'])+'/'+str(m['pairs']):>16} "
          f"{str(m['preserved'])+'/'+str(m['pairs']):>12} "
          f"{str(m['upper_escapes_lower'])+'/'+str(m['upper_pairs']):>16}")


  dim   boundary         REVERSED    preserved    upper escapes
    8        e_4              6/6          0/6             0/16
   16        e_8            42/42         0/42             0/64
   32       e_16          210/210        0/210            0/256
   64       e_32          930/930        0/930           0/1024


**Order-reversal is total and exception-free at every level tested, and the upper
half never lands in itself.**

This is what the doubling rule says algebraically: `(0,b)(0,d) = (-d* b, 0)`. The
product of two upper elements falls into the *lower* half, with the order reversed
and a conjugate applied. Order-reversal is the definition of an **anti**-automorphism
— a reflection that flips handedness.

So the answer to the originating question is: **it mirrors, and it is chiral.**
Not the non-chiral reflection that was hoped for.


## 2.2 The orphans — ORIGINATING OBSERVATION

Which indices belong to no zero-divisor plane?


In [4]:
census = {}
for lv in (3, 4, 5):
    dim, H, rows = zd_census(lv)
    census[dim] = rows
    print(f"dim {dim:>3}   ZD diagonals {len(rows):>5}   orphans {orphans_from(dim, rows)}")


dim   8   ZD diagonals     0   orphans [0, 1, 2, 3, 4, 5, 6, 7]
dim  16   ZD diagonals    84   orphans [0, 8]


dim  32   ZD diagonals   588   orphans [0, 16]


`dim = 8` returns every index because the octonions are a **division algebra** —
no zero divisors exist, so the question is vacuous there. This was recorded as a
scope condition in `01_predictions` *before* the result was produced.

From `dim = 16` on, the orphans are `[0, dim/2]`:

- **`e_0`** — the identity. `box_kite.e0_is_outside()` verifies `[e_0, ., .] = 0`:
  it generates the boundary and does not live on it.
- **`e_{dim/2}`** — the generator introduced by *this* doubling.

Note what does **not** happen: `e_8` stops being an orphan once it is embedded in
`dim = 32`. The boundary does not accumulate. There is exactly **one** boundary per
level, and it moves.


## 2.3 The balance — ORIGINATING OBSERVATION

Census by location and nullity at `dim = 32`.


In [5]:
rows = census[32]
locs = ['lower', 'cross', 'upper']
nulls = sorted({r['nullity'] for r in rows})
print(f"{'nullity':>8}" + ''.join(f"{l:>9}" for l in locs) + f"{'total':>9}")
for n in nulls:
    cnt = [sum(1 for r in rows if r['nullity'] == n and r['loc'] == l) for l in locs]
    print(f"{n:>8}" + ''.join(f"{c:>9}" for c in cnt) + f"{sum(cnt):>9}")
tot = [sum(1 for r in rows if r['loc'] == l) for l in locs]
print(f"{'total':>8}" + ''.join(f"{c:>9}" for c in tot) + f"{sum(tot):>9}")
print()
print('as multiples of 84:', {l: c / 84 for l, c in zip(locs, tot)}, '| total', sum(tot) / 84)


 nullity    lower    cross    upper    total
       4        0      168        0      168
       8       84        0       84      168
      12        0      252        0      252
   total       84      420       84      588

as multiples of 84: {'lower': 1.0, 'cross': 5.0, 'upper': 1.0} | total 7.0


Two things fall out.

**The balance is exact.** `84` lower, `84` upper. The two halves carry the same
number of internal zero divisors.

**Nullity separates location perfectly.** Non-crossing divisors always have nullity
`8`; crossing divisors never do. The kernel dimension alone tells you whether a
divisor stays inside a half or spans the boundary — with no exceptions in either
direction.

That is the same internal/external discriminator the `angular_rank` engine uses,
appearing at the level of the algebra rather than the instrument.


## 2.4 The asymmetry — ORIGINATING OBSERVATION

The halves balance in census. Do they balance in structure?


In [6]:
tab, dim = cd(5); H = dim // 2
lower, upper = list(range(H)), list(range(H, dim))
for name, idx in (('LOWER (a,0)', lower), ('UPPER (0,b)', upper)):
    inside = sum(1 for i in idx for j in idx if tab[(i, j)][1] in idx)
    print(f"{name}: {inside}/{len(idx)**2} products stay inside   CLOSED = {inside == len(idx)**2}")

def swap(v, H): return list(v[H:]) + list(v[:H])

def mul(tab, dim, x, y):
    out = [0.0]*dim
    for i in range(dim):
        if x[i] == 0.0: continue
        for j in range(dim):
            if y[j] == 0.0: continue
            s, k = tab[(i, j)]; out[k] += s*x[i]*y[j]
    return out

bad = 0
for i in range(dim):
    for j in range(dim):
        x = [0.0]*dim; x[i] = 1.0
        y = [0.0]*dim; y[j] = 1.0
        if not np.allclose(swap(mul(tab, dim, x, y), H), mul(tab, dim, swap(x, H), swap(y, H))):
            bad += 1
print(f"exchange (a,b)->(b,a) violations: {bad}/{dim*dim}   AUTOMORPHISM = {bad == 0}")


LOWER (a,0): 256/256 products stay inside   CLOSED = True
UPPER (0,b): 0/256 products stay inside   CLOSED = False
exchange (a,b)->(b,a) violations: 1024/1024   AUTOMORPHISM = False


**The lower half is closed. The upper half is closed 0 out of 256 times — never.**
And the exchange map fails on every single basis pair.

So the two halves are **balanced in census and not interchangeable in structure**.
The upper half has the shape of an algebra and none of the closure of one: it cannot
compose two of its own elements without the result appearing in the lower half.

⚠ **Interpretation is not licensed here.** This rules out any reading in which the
two halves are structurally equal peers. It does not establish what either half *is*.

---

*Next: `03_results.ipynb` — the pre-registered predictions, scored.*
